# Step 5 — TTC Estimation using UKF + CTRV (Upgraded) ✅

| | |
|---|---|
| **Input** | `output/step_3/{lidar,radar,camera}/track_*.json`, `output/step_4/fused_tracks_all.csv`, `output/step_1/lidar/<sample>/lidar_meta.json` (for ego position) |
| **Outputs** | `output/step_5/ttc_lidar.csv`, `ttc_radar.csv`, `ttc_camera.csv`, `ttc_fused.csv` — same schema, genuinely comparable |
| **Used by** | Step 6 (visualization + evaluation metrics) |

---

### What was found and fixed

1. **A real UKF existed but wasn't used.** Cell 0 of your original had a correct Merwe-sigma-point UKF with CTRV process/measurement models. Cell 1 — the one that actually ran — used a function literally named `ukf_ctrv_velocity` whose own docstring says "Placeholder... for now, use smoothed gradient as stand-in." This matches your dissertation's own stated limitation exactly. This version wires in the real UKF from Cell 0.
2. **Ego assumed to be at the origin (0,0).** True under the old sensor-local pipeline, but no longer true now that Steps 3.1-3.3 output global-frame positions. Distance and closing speed are now computed against the ego vehicle's actual global position at each sample.
3. **frame_dt was wrong (0.05s / 0.1s assumed) — actual keyframe rate is 2Hz (0.5s).** Replaced with real per-step timestamp differences.
4. **Closing speed didn't account for the ego's own motion.** In global frame, both the object and the ego vehicle move — true closing speed needs object_velocity minus ego_velocity, not just the object's velocity. Ego velocity is now estimated from consecutive ego positions.
5. **No single-sensor baselines.** Added — the same corrected function now runs on LiDAR-only, Radar-only, and Camera-only tracks in addition to the fused tracks, so the evaluation comparison is apples-to-apples.

### Still missing (flagging, not fixing here)
Ground-truth TTC for MAE/RMSE isn't computed anywhere in what's been shared so far. Check if it's in Step 6 — if not, it needs to be built by matching tracks to nuScenes' annotated boxes.

In [1]:
# CELL 1 — Verify config.py exists

from pathlib import Path

if not Path("config.py").exists():
    raise FileNotFoundError("config.py not found. Copy it from the repo root.")

from config import STEP0_DIR, STEP1_DIR, STEP3_DIR, STEP4_DIR, STEP5_DIR

LIDAR_TRACKS_DIR  = STEP3_DIR / "lidar"
RADAR_TRACKS_DIR  = STEP3_DIR / "radar"
CAMERA_TRACKS_DIR = STEP3_DIR / "camera"
LIDAR_META_DIR    = STEP1_DIR / "lidar"
FUSED_CSV_PATH    = STEP4_DIR / "fused_tracks_all.csv"
STEP5_DIR.mkdir(parents=True, exist_ok=True)

for p, name in [(LIDAR_TRACKS_DIR, "Step 3.1"), (RADAR_TRACKS_DIR, "Step 3.2"),
                 (CAMERA_TRACKS_DIR, "Step 3.3"), (FUSED_CSV_PATH, "Step 4")]:
    if not p.exists():
        raise FileNotFoundError(f"{name} output not found at {p} — run that step first.")

print("config OK. All inputs found.")

config.py loaded. PROJECT_ROOT = F:\Sensor fusion Research
DATA_ROOT   = F:\Sensor fusion Research\DATA SET\archive
OUTPUT_ROOT = F:\Sensor fusion Research\output
config OK. All inputs found.


In [2]:
# CELL 2 — Build ego position + velocity lookup per sample
# This replaces the "ego is at (0,0)" assumption

import json
import numpy as np

with open(STEP0_DIR / "samples_index.json") as f:
    samples_index = json.load(f)

ego_pos_by_sample = {}
for sample_id, info in samples_index.items():
    meta_path = LIDAR_META_DIR / sample_id / "lidar_meta.json"
    if not meta_path.exists():
        continue
    with open(meta_path) as f:
        meta = json.load(f)
    ego_pos_by_sample[sample_id] = {
        "pos": np.array(meta["ego_pose"]["translation"][:2]),
        "timestamp": info["timestamp_us"]
    }

sorted_samples = sorted(ego_pos_by_sample.keys(), key=lambda s: ego_pos_by_sample[s]["timestamp"])
ego_vel_by_sample = {}
for i in range(1, len(sorted_samples)):
    s0, s1 = sorted_samples[i-1], sorted_samples[i]
    dt = (ego_pos_by_sample[s1]["timestamp"] - ego_pos_by_sample[s0]["timestamp"]) / 1e6
    if dt <= 0:
        continue
    v = (ego_pos_by_sample[s1]["pos"] - ego_pos_by_sample[s0]["pos"]) / dt
    ego_vel_by_sample[s1] = v
if len(sorted_samples) > 1:
    ego_vel_by_sample[sorted_samples[0]] = ego_vel_by_sample.get(sorted_samples[1], np.array([0.0, 0.0]))

print(f"Ego position/velocity computed for {len(ego_pos_by_sample)} samples.")
if ego_vel_by_sample:
    print(f"Mean ego speed: {np.mean([np.linalg.norm(v) for v in ego_vel_by_sample.values()]):.2f} m/s")

Ego position/velocity computed for 404 samples.
Mean ego speed: 5.53 m/s


In [3]:
# CELL 3 — Real UKF + CTRV (from your Cell 0 — kept as-is, it was correct)

def merwe_sigma_points(x, P, alpha=1e-3, beta=2.0, kappa=0.0):
    n = x.size
    lam = alpha**2 * (n + kappa) - n
    c = n + lam
    U = np.linalg.cholesky(P * c)
    Wm = np.full(2*n+1, 1/(2*c)); Wc = np.full(2*n+1, 1/(2*c))
    Wm[0] = lam / c; Wc[0] = lam / c + (1 - alpha**2 + beta)
    sigmas = np.zeros((2*n+1, n))
    sigmas[0] = x
    for i in range(n):
        sigmas[i+1]   = x + U[:, i]
        sigmas[n+i+1] = x - U[:, i]
    return sigmas, Wm, Wc


def ctrv_process(sigma, dt):
    """State: [x, y, v, yaw, yaw_rate] — this IS the standard CTRV form.
    Note: dissertation text describes the state as [x,y,vx,vy,psi] -- the
    code's [x,y,v,yaw,yaw_rate] is actually the more correct CTRV
    parameterization; vx,vy are derived below for reporting to match the
    dissertation's language. Worth a one-line clarification in your defense.
    """
    x, y, v, yaw, yawd = sigma
    if abs(yawd) > 1e-6:
        x_p = x + v/yawd * (np.sin(yaw + yawd*dt) - np.sin(yaw))
        y_p = y + v/yawd * (-np.cos(yaw + yawd*dt) + np.cos(yaw))
    else:
        x_p = x + v * np.cos(yaw) * dt
        y_p = y + v * np.sin(yaw) * dt
    return np.array([x_p, y_p, v, yaw + yawd*dt, yawd])


def ukf_predict(x, P, Q, dt):
    sigmas, Wm, Wc = merwe_sigma_points(x, P)
    sigmas_f = np.array([ctrv_process(s, dt) for s in sigmas])
    x_pred = np.sum(Wm[:, None] * sigmas_f, axis=0)
    P_pred = Q.copy()
    for i in range(sigmas_f.shape[0]):
        y = sigmas_f[i] - x_pred
        y[3] = (y[3] + np.pi) % (2*np.pi) - np.pi
        P_pred += Wc[i] * np.outer(y, y)
    return x_pred, P_pred, sigmas_f, Wm, Wc


def ukf_update(x_pred, P_pred, sigmas_f, Wm, Wc, z, R):
    Z = np.array([s[:2] for s in sigmas_f])
    z_pred = np.sum(Wm[:, None] * Z, axis=0)
    S = R.copy()
    for i in range(Z.shape[0]):
        dz = Z[i] - z_pred
        S += Wc[i] * np.outer(dz, dz)
    n = x_pred.size
    Tc = np.zeros((n, 2))
    for i in range(Z.shape[0]):
        dx = sigmas_f[i] - x_pred
        dx[3] = (dx[3] + np.pi) % (2*np.pi) - np.pi
        dz = Z[i] - z_pred
        Tc += Wc[i] * np.outer(dx, dz)
    K = Tc @ np.linalg.inv(S)
    dz = z - z_pred
    x_upd = x_pred + K @ dz
    P_upd = P_pred - K @ S @ K.T
    return x_upd, P_upd


print("UKF core functions loaded (unchanged from your original Cell 0 — it was correct).")

UKF core functions loaded (unchanged from your original Cell 0 — it was correct).


In [4]:
# CELL 4 — Unified TTC runner, used identically for all 4 datasets
# FIXED: real per-step dt, ego at its actual global position + velocity,
# real UKF for tracks with 3+ points, direct calc for 2-point tracks

import pandas as pd


# def compute_ttc(px, py, vx, vy, sample_id):
#     """Distance and closing speed relative to the ego's ACTUAL global
#     position and velocity at this sample — not a fixed origin."""
#     if sample_id not in ego_pos_by_sample:
#         return np.nan, np.nan, np.nan
#     ego_pos = ego_pos_by_sample[sample_id]["pos"]
#     ego_vel = ego_vel_by_sample.get(sample_id, np.array([0.0, 0.0]))

#     rel_pos = np.array([px, py]) - ego_pos
#     dist = np.linalg.norm(rel_pos)
#     if dist <= 1e-6:
#         return dist, 0.0, np.inf

#     los = rel_pos / dist
#     rel_vel = np.array([vx, vy]) - ego_vel
#     closing_speed = -np.dot(rel_vel, los)

#     ttc = dist / closing_speed if closing_speed > 1e-6 else np.inf
#     return dist, closing_speed, ttc

MIN_CLOSING_SPEED = 0.3   # m/s — below this, not meaningfully "approaching"
MAX_VALID_TTC = 60.0      # seconds — beyond this, not collision-relevant

def compute_ttc(px, py, vx, vy, sample_id):
    if sample_id not in ego_pos_by_sample:
        return np.nan, np.nan, np.nan
    ego_pos = ego_pos_by_sample[sample_id]["pos"]
    ego_vel = ego_vel_by_sample.get(sample_id, np.array([0.0, 0.0]))

    rel_pos = np.array([px, py]) - ego_pos
    dist = np.linalg.norm(rel_pos)
    if dist <= 1e-6:
        return dist, 0.0, np.inf

    los = rel_pos / dist
    rel_vel = np.array([vx, vy]) - ego_vel
    closing_speed = -np.dot(rel_vel, los)

    if closing_speed < MIN_CLOSING_SPEED:
        return dist, closing_speed, np.inf        # not meaningfully approaching

    ttc = dist / closing_speed
    if ttc > MAX_VALID_TTC:
        return dist, closing_speed, np.inf          # too far out to be meaningful

    return dist, closing_speed, ttc


def run_ttc_pipeline(df, min_points=2):
    """df must have columns: fused_id, sample_id, timestamp, x, y"""
    df = df.sort_values(['fused_id', 'timestamp']).reset_index(drop=True)
    results = []

    for fid, g in df.groupby('fused_id'):
        g = g.sort_values('timestamp').reset_index(drop=True)
        if len(g) < min_points:
            continue

        if len(g) == 2:
            dt = (g.loc[1, 'timestamp'] - g.loc[0, 'timestamp']) / 1e6
            if dt <= 0:
                continue
            vx = (g.loc[1, 'x'] - g.loc[0, 'x']) / dt
            vy = (g.loc[1, 'y'] - g.loc[0, 'y']) / dt
            for i in range(2):
                dist, cs, ttc = compute_ttc(g.loc[i, 'x'], g.loc[i, 'y'], vx, vy, g.loc[i, 'sample_id'])
                results.append({"fused_id": fid, "sample_id": g.loc[i, 'sample_id'],
                 "timestamp": g.loc[i, 'timestamp'],          # ← add this line
                 "x": g.loc[i, 'x'], "y": g.loc[i, 'y'],
                 "vx": vx, "vy": vy, "distance": dist,
                 "closing_speed": cs, "ttc": ttc, "method": "direct_2pt"})
            continue

        dt0 = (g.loc[1, 'timestamp'] - g.loc[0, 'timestamp']) / 1e6
        if dt0 <= 0:
            continue
        dx = (g.loc[1, 'x'] - g.loc[0, 'x']) / dt0
        dy = (g.loc[1, 'y'] - g.loc[0, 'y']) / dt0
        v0 = np.hypot(dx, dy)
        yaw0 = np.arctan2(dy, dx) if v0 > 1e-6 else 0.0

        x = np.array([g.loc[0, 'x'], g.loc[0, 'y'], v0, yaw0, 0.0])
        P = np.diag([1.0, 1.0, 5.0, 0.5, 0.5])
        Q = np.diag([0.2]*5)
        R = np.diag([0.5, 0.5])

        prev_timestamp = g.loc[0, 'timestamp']
        for i, row in g.iterrows():
            dt = max((row['timestamp'] - prev_timestamp) / 1e6, 1e-3)
            prev_timestamp = row['timestamp']

            z = np.array([row['x'], row['y']])
            x_pred, P_pred, sigmas_f, Wm, Wc = ukf_predict(x, P, Q, dt)
            x, P = ukf_update(x_pred, P_pred, sigmas_f, Wm, Wc, z, R)

            v, yaw = x[2], x[3]
            vx, vy = v * np.cos(yaw), v * np.sin(yaw)

            dist, cs, ttc = compute_ttc(x[0], x[1], vx, vy, row['sample_id'])
            results.append({"fused_id": fid, "sample_id": row['sample_id'],
                 "timestamp": row['timestamp'],                # ← add this line
                 "x": x[0], "y": x[1], "vx": vx, "vy": vy,
                 "distance": dist, "closing_speed": cs, "ttc": ttc, "method": "ukf_ctrv"})

    return pd.DataFrame(results)


print("run_ttc_pipeline() defined — same function used for all 4 datasets below.")

run_ttc_pipeline() defined — same function used for all 4 datasets below.


In [5]:
# CELL 5 — Load all 4 datasets into the same common schema

def load_single_sensor_tracks(track_dir):
    """Step 3.1/3.2 tuple format: (sample_id, timestamp, [x,y,z])"""
    rows = []
    for file in track_dir.glob("track_*.json"):
        with open(file) as f:
            points = json.load(f)
        for sample_id, timestamp, pos in points:
            rows.append({"fused_id": file.stem, "sample_id": sample_id,
                         "timestamp": timestamp, "x": pos[0], "y": pos[1]})
    return pd.DataFrame(rows)


def load_camera_tracks_df(track_dir):
    """Step 3.3 dict/trajectory format"""
    rows = []
    for file in track_dir.glob("track_*.json"):
        with open(file) as f:
            data = json.load(f)
        for pt in data["trajectory"]:
            ts = samples_index.get(pt["sample_id"], {}).get("timestamp_us")
            rows.append({"fused_id": data["track_id"], "sample_id": pt["sample_id"],
                         "timestamp": ts, "x": pt["pos"][0], "y": pt["pos"][1]})
    return pd.DataFrame(rows)


lidar_df  = load_single_sensor_tracks(LIDAR_TRACKS_DIR)
radar_df  = load_single_sensor_tracks(RADAR_TRACKS_DIR)
camera_df = load_camera_tracks_df(CAMERA_TRACKS_DIR)
fused_df  = pd.read_csv(FUSED_CSV_PATH)[["fused_id", "sample_id", "timestamp", "x", "y"]]

print(f"LiDAR : {len(lidar_df)} rows, {lidar_df['fused_id'].nunique()} tracks")
print(f"Radar : {len(radar_df)} rows, {radar_df['fused_id'].nunique()} tracks")
print(f"Camera: {len(camera_df)} rows, {camera_df['fused_id'].nunique()} tracks")
print(f"Fused : {len(fused_df)} rows, {fused_df['fused_id'].nunique()} tracks")

LiDAR : 16179 rows, 3134 tracks
Radar : 5490 rows, 1693 tracks
Camera: 4399 rows, 1131 tracks
Fused : 20301 rows, 4378 tracks


In [6]:
# CELL 6 — Run the SAME corrected TTC pipeline on all 4 datasets

datasets = {"lidar": lidar_df, "radar": radar_df, "camera": camera_df, "fused": fused_df}
ttc_results = {}

for name, df in datasets.items():
    print(f"Running TTC for: {name}")
    result_df = run_ttc_pipeline(df)
    ttc_results[name] = result_df

    out_path = STEP5_DIR / f"ttc_{name}.csv"
    result_df.to_csv(out_path, index=False)
    print(f"  {len(result_df)} TTC rows saved to {out_path}")

Running TTC for: lidar


  16179 TTC rows saved to F:\Sensor fusion Research\output\step_5\ttc_lidar.csv
Running TTC for: radar


  5490 TTC rows saved to F:\Sensor fusion Research\output\step_5\ttc_radar.csv
Running TTC for: camera


  4399 TTC rows saved to F:\Sensor fusion Research\output\step_5\ttc_camera.csv
Running TTC for: fused


  20301 TTC rows saved to F:\Sensor fusion Research\output\step_5\ttc_fused.csv


In [7]:
# CELL 7 — Quick comparison summary (full evaluation table belongs in Step 6)

summary_rows = []
for name, df in ttc_results.items():
    valid_ttc = df[df["ttc"].notna() & np.isfinite(df["ttc"])]
    summary_rows.append({
        "sensor": name,
        "total_points": len(df),
        "valid_ttc_count": len(valid_ttc),
        "percent_valid": round(len(valid_ttc) / len(df) * 100, 1) if len(df) > 0 else 0.0,
        "mean_ttc": round(valid_ttc["ttc"].mean(), 2) if len(valid_ttc) > 0 else None,
        "median_ttc": round(valid_ttc["ttc"].median(), 2) if len(valid_ttc) > 0 else None,
    })

summary_df = pd.DataFrame(summary_rows)
print("Quick comparison (full MAE/RMSE against ground truth belongs in Step 6):")
display(summary_df)

summary_df.to_csv(STEP5_DIR / "ttc_quick_summary.csv", index=False)
print(f"Saved: {STEP5_DIR / 'ttc_quick_summary.csv'}")
print("Reminder: ground-truth TTC (needed for MAE/RMSE) is not computed anywhere")
print("in the notebooks reviewed so far. Check Step 6 -- if it's not there either,")
print("it needs to be built by matching tracks to nuScenes annotated boxes.")

Quick comparison (full MAE/RMSE against ground truth belongs in Step 6):


,sensor,total_points,valid_ttc_count,percent_valid,mean_ttc,median_ttc
0,lidar,16179,6823,42.2,10.35,5.58
1,radar,5490,2203,40.1,11.41,7.24
2,camera,4399,2284,51.9,8.54,5.28
3,fused,20301,8511,41.9,10.99,6.17


Saved: F:\Sensor fusion Research\output\step_5\ttc_quick_summary.csv
Reminder: ground-truth TTC (needed for MAE/RMSE) is not computed anywhere
in the notebooks reviewed so far. Check Step 6 -- if it's not there either,
it needs to be built by matching tracks to nuScenes annotated boxes.
